In [ ]:
!pip install playwright nest_asyncio pandas
!playwright install chromium
!playwright install-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 16.5 MB/s eta 0:00:00
(node:2901) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
167.3 MiB [] 0% 344.7s167.3 MiB [] 0% 290.2s167.3 MiB [] 0% 300.3s167.3 MiB [] 0% 1064.7s167.3 MiB [] 0% 876.1s167.3 MiB [] 0% 1219.1s167.3 MiB [] 0% 1067.5s167.3 MiB [] 0% 960.7s167.3 MiB [] 0% 879.2s167.3 MiB [] 0% 815.9s167.3 MiB [] 0% 837.0s167.3 MiB [] 0% 789.0s167.3 MiB [] 0% 749.0s167.3 MiB [] 0% 709.4s167.3 MiB [] 0% 680.8s167.3 MiB [] 0% 656.0s167.3 MiB [] 0% 633.7s167.3 MiB [] 0% 611.4s167.3 MiB [] 0% 591.1s167.3 MiB [] 0% 572.8s167.3 MiB [] 0% 554.3s167.3 MiB [] 0% 523.0s167.3 MiB [] 0% 502.3s167.3 MiB [] 0% 474.4s167.3 MiB [] 0% 450.3s167.3 MiB [] 0% 429.4s167.3 MiB [] 0% 411.0s167.3 MiB [] 0% 394.7s16

In [ ]:
import asyncio
import nest_asyncio
import pandas as pd
from playwright.async_api import async_playwright

nest_asyncio.apply()

urls = [
"https://www.threads.com/@drg.silva/post/DWssnN7DrFR",
"https://www.threads.com/@asgaryuk/post/DU6yiDRk8Km",
"https://www.threads.com/@doktercamo/post/DWK2Iawk_6i",
"https://www.threads.com/@dr.doddyrizqi/post/DVxezlnjxng",
"https://www.threads.com/@st.antonius678/post/DWDcEbpEZwc",
"https://www.threads.com/@fadly_aja.ya/post/DRnwTJ1AfbZ",
"https://www.threads.com/@bjornnesy/post/DUU0sFIkpV_",
"https://www.threads.com/@farahdhitafff/post/DWSb2haD1V4",
"https://www.threads.com/@lillasyifa/post/DUa6HEGkpww",
"https://www.threads.com/@ratih.anggraaini/post/DUYk2lYk9NW",
]


async def scrape_threads(urls):

    results = []

    async with async_playwright() as p:

        browser = await p.chromium.launch(
            headless=True,
            args=["--no-sandbox", "--disable-setuid-sandbox"]
        )

        page = await browser.new_page()

        for url in urls:

            print("Scraping:", url)

            try:

                await page.goto(url, timeout=60000)
                await page.wait_for_timeout(5000)

                # scroll load komentar
                for _ in range(8):
                    await page.mouse.wheel(0, 4000)
                    await page.wait_for_timeout(2000)

                elements = await page.query_selector_all("article")

                for el in elements:

                    text = await el.inner_text()

                    results.append({
                        "post_url": url,
                        "comment": text
                    })

            except Exception as e:
                print("Error:", e)

        await browser.close()

    return results


data = asyncio.run(scrape_threads(urls))

df = pd.DataFrame(data)

df.head()

Scraping: https://www.threads.com/@drg.silva/post/DWssnN7DrFR
Scraping: https://www.threads.com/@asgaryuk/post/DU6yiDRk8Km
Scraping: https://www.threads.com/@doktercamo/post/DWK2Iawk_6i
Scraping: https://www.threads.com/@dr.doddyrizqi/post/DVxezlnjxng
Scraping: https://www.threads.com/@st.antonius678/post/DWDcEbpEZwc
Scraping: https://www.threads.com/@fadly_aja.ya/post/DRnwTJ1AfbZ
Scraping: https://www.threads.com/@bjornnesy/post/DUU0sFIkpV_
Scraping: https://www.threads.com/@farahdhitafff/post/DWSb2haD1V4
Scraping: https://www.threads.com/@lillasyifa/post/DUa6HEGkpww
Scraping: https://www.threads.com/@ratih.anggraaini/post/DUYk2lYk9NW


""


In [ ]:
async def scrape_threads(urls):

    results = []

    async with async_playwright() as p:

        browser = await p.chromium.launch(
            headless=True,
            args=[
                "--no-sandbox",
                "--disable-setuid-sandbox",
                "--disable-dev-shm-usage"
            ]
        )

        page = await browser.new_page()

        for url in urls:

            print("Scraping:", url)

            await page.goto(url)
            await page.wait_for_timeout(6000)

            # scroll banyak supaya komentar muncul
            for _ in range(15):
                await page.mouse.wheel(0, 5000)
                await page.wait_for_timeout(1500)

            elements = await page.query_selector_all("div[data-pressable-container]")

            print("Elemen ditemukan:", len(elements))

            for el in elements:

                try:
                    text = await el.inner_text()

                    if len(text) > 20:   # filter supaya bukan UI
                        results.append({
                            "post_url": url,
                            "comment": text
                        })

                except:
                    pass

        await browser.close()

    return results

In [ ]:
data = asyncio.run(scrape_threads(urls))

print("Total komentar:", len(data))

df = pd.DataFrame(data)

df.head(50)

Scraping: https://www.threads.com/@drg.silva/post/DWssnN7DrFR
Elemen ditemukan: 24
Scraping: https://www.threads.com/@asgaryuk/post/DU6yiDRk8Km
Elemen ditemukan: 21
Scraping: https://www.threads.com/@doktercamo/post/DWK2Iawk_6i
Elemen ditemukan: 35
Scraping: https://www.threads.com/@dr.doddyrizqi/post/DVxezlnjxng
Elemen ditemukan: 12
Scraping: https://www.threads.com/@st.antonius678/post/DWDcEbpEZwc
Elemen ditemukan: 29
Scraping: https://www.threads.com/@fadly_aja.ya/post/DRnwTJ1AfbZ
Elemen ditemukan: 15
Scraping: https://www.threads.com/@bjornnesy/post/DUU0sFIkpV_
Elemen ditemukan: 3
Scraping: https://www.threads.com/@farahdhitafff/post/DWSb2haD1V4
Elemen ditemukan: 45
Scraping: https://www.threads.com/@lillasyifa/post/DUa6HEGkpww
Elemen ditemukan: 32
Scraping: https://www.threads.com/@ratih.anggraaini/post/DUYk2lYk9NW
Elemen ditemukan: 30
Total komentar: 246


,post_url,comment
0,https://www.threads.com/@drg.silva/post/DWssnN...,drg.silva\n3h\n“Bau khas” pada penderita diabe...
1,https://www.threads.com/@drg.silva/post/DWssnN...,kons_ngete\n3h\nReplying to @drg.silva\nDok ke...
2,https://www.threads.com/@drg.silva/post/DWssnN...,"Pinned\nsi.troy\n2h\nBaunya kayak apa sih, Dok..."
3,https://www.threads.com/@drg.silva/post/DWssnN...,"nzshiro\n1h\nNah ini. Kalo dalam tekpang, keto..."
4,https://www.threads.com/@drg.silva/post/DWssnN...,cintapertama9535\n43m\nDok suami saya Diabetes...
5,https://www.threads.com/@drg.silva/post/DWssnN...,februaritaa\n1h\nReal ya aku tadi kontrol poli...
6,https://www.threads.com/@drg.silva/post/DWssnN...,aarkidhya_orchids\n14m\noh..mungkin begini ya ...
7,https://www.threads.com/@drg.silva/post/DWssnN...,klinik_hewan_bandung\n2h\ndok itu sama kayak g...
8,https://www.threads.com/@drg.silva/post/DWssnN...,drg.silva\n2h\n·\nAuthor\nLebih tepatnya ketoa...
9,https://www.threads.com/@drg.silva/post/DWssnN...,klinik_hewan_bandung\n2h\nowhh ok. mengerti se...


In [ ]:
df.to_csv("threads_diabetes_05Apr.csv", index=False)

In [ ]:
display(df)

,post_url,comment
0,https://www.threads.com/@drg.silva/post/DWssnN...,drg.silva\n3h\n“Bau khas” pada penderita diabe...
1,https://www.threads.com/@drg.silva/post/DWssnN...,kons_ngete\n3h\nReplying to @drg.silva\nDok ke...
2,https://www.threads.com/@drg.silva/post/DWssnN...,"Pinned\nsi.troy\n2h\nBaunya kayak apa sih, Dok..."
3,https://www.threads.com/@drg.silva/post/DWssnN...,"nzshiro\n1h\nNah ini. Kalo dalam tekpang, keto..."
4,https://www.threads.com/@drg.silva/post/DWssnN...,cintapertama9535\n43m\nDok suami saya Diabetes...
...,...,...
241,https://www.threads.com/@ratih.anggraaini/post...,primanez\n02/06/26\nWihh\n1\n1
242,https://www.threads.com/@ratih.anggraaini/post...,henky_chan_story\n02/06/26\nHahaha
243,https://www.threads.com/@ratih.anggraaini/post...,delldjohan\n02/06/26\nAlhamdulillah sy 54 gak ...
244,https://www.threads.com/@ratih.anggraaini/post...,uniiemurnie\n02/07/26\nHuhuhu aku baru 33 tahu...


In [ ]:
import re
import pandas as pd

def parse_threads_comment(comment_string):
    username = ""
    date_str = ""
    comment_text = ""

    lines = comment_string.split('\n')

    if lines:
        username = lines[0].strip()

        # Join the rest of the lines to search for date and comment
        remaining_text_raw = "\n".join(lines[1:])

        # Regex to find a date pattern like MM/DD/YY
        date_match = re.search(r'(\d{2}/\d{2}/\d{2})', remaining_text_raw)

        if date_match:
            date_str = date_match.group(1)
            # Split the string at the date and take the part after it
            parts_around_date = remaining_text_raw.split(date_str, 1)
            temp_comment = parts_around_date[1].strip() if len(parts_around_date) > 1 else ""

            # Clean up known UI elements: '·', 'Author', 'Translate', and engagement numbers (e.g., 6.2K, 911, 1.5K, 2.5K)
            comment_text = re.sub(r'\s*·\s*Author|\s*Translate|\d+(\.\d+)?[KMB]?\s*', ' ', temp_comment)
            comment_text = re.sub(r'\s+', ' ', comment_text).strip() # Replace multiple spaces with a single space
            comment_text = re.sub(r'^\s*-\s*', '', comment_text) # Remove leading '- ' if any

        else:
            # If no date found, assume the entire remaining_text_raw is the comment
            comment_text = re.sub(r'\s*·\s*Author|\s*Translate|\d+(\.\d+)?[KMB]?\s*', ' ', remaining_text_raw)
            comment_text = re.sub(r'\s+', ' ', comment_text).strip()

    return username, date_str, comment_text

# Apply the parsing function to the 'comment' column
df[['username', 'date_raw', 'comment_new']] = df['comment'].apply(
    lambda x: pd.Series(parse_threads_comment(x))
)

# Convert 'date_raw' to datetime objects. Handle errors if any.
df['date'] = pd.to_datetime(df['date_raw'], format='%m/%d/%y', errors='coerce')

# Drop the intermediate 'date_raw' and original 'comment' columns
df_cleaned = df.drop(columns=['date_raw', 'comment'])

# Rename 'comment_new' to 'comment' and reorder columns for readability
df_cleaned = df_cleaned.rename(columns={'comment_new': 'comment'})[['post_url', 'username', 'date', 'comment']]

print("Cleaned DataFrame info:")
df_cleaned.info()
print("\nFirst 5 rows of the cleaned DataFrame:")
display(df_cleaned.head())

Cleaned DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 246 entries, 0 to 245
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   post_url  246 non-null    object        
 1   username  246 non-null    object        
 2   date      220 non-null    datetime64[ns]
 3   comment   246 non-null    object        
dtypes: datetime64[ns](1), object(3)
memory usage: 7.8+ KB

First 5 rows of the cleaned DataFrame:


,post_url,username,date,comment
0,https://www.threads.com/@drg.silva/post/DWssnN...,drg.silva,NaT,h “Bau khas” pada penderita diabetes disebut j...
1,https://www.threads.com/@drg.silva/post/DWssnN...,kons_ngete,NaT,h Replying to @drg.silva Dok kenapa orang meng...
2,https://www.threads.com/@drg.silva/post/DWssnN...,Pinned,NaT,"si.troy h Baunya kayak apa sih, Dok? Gak kebay..."
3,https://www.threads.com/@drg.silva/post/DWssnN...,nzshiro,NaT,"h Nah ini. Kalo dalam tekpang, keton adalah pr..."
4,https://www.threads.com/@drg.silva/post/DWssnN...,cintapertama9535,NaT,"m Dok suami saya Diabetes, bau mulutnya ketika..."


In [ ]:
# Save the cleaned DataFrame to a new CSV file
df_cleaned.to_csv("threads_diabetes_cleaned_05Apr.csv", index=False)
print("Cleaned data saved to 'threads_diabetes_cleaned_29Mar.csv'")

Cleaned data saved to 'threads_diabetes_cleaned_29Mar.csv'


In [ ]:
df_cleaned = df_cleaned.drop_duplicates(subset=['comment'], keep='first')

print("Cleaned DataFrame after removing duplicate comments:")
df_cleaned.info()
display(df_cleaned.head())

Cleaned DataFrame after removing duplicate comments:
<class 'pandas.core.frame.DataFrame'>
Index: 245 entries, 0 to 245
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   post_url  245 non-null    object        
 1   username  245 non-null    object        
 2   date      219 non-null    datetime64[ns]
 3   comment   245 non-null    object        
dtypes: datetime64[ns](1), object(3)
memory usage: 9.6+ KB


,post_url,username,date,comment
0,https://www.threads.com/@drg.silva/post/DWssnN...,drg.silva,NaT,h “Bau khas” pada penderita diabetes disebut j...
1,https://www.threads.com/@drg.silva/post/DWssnN...,kons_ngete,NaT,h Replying to @drg.silva Dok kenapa orang meng...
2,https://www.threads.com/@drg.silva/post/DWssnN...,Pinned,NaT,"si.troy h Baunya kayak apa sih, Dok? Gak kebay..."
3,https://www.threads.com/@drg.silva/post/DWssnN...,nzshiro,NaT,"h Nah ini. Kalo dalam tekpang, keton adalah pr..."
4,https://www.threads.com/@drg.silva/post/DWssnN...,cintapertama9535,NaT,"m Dok suami saya Diabetes, bau mulutnya ketika..."


In [ ]:
# Save the cleaned DataFrame to a new CSV file
df_cleaned.to_csv("threads_diabetes_cleaned_05Apr.csv", index=False)
print("Cleaned data saved")

Cleaned data saved
